# Projeto Fictus | Análise Logística - ETL Pipeline

---

## Contexto do Bloco

A Fase 1 (analise de vendas) estabeleceu a viabilidade comercial do ativo, mas identificou a logística como uma condicionante crítica para a sustentabilidade do negócio. Para que a auditoria logística seja precisa, não basta olhar para o custo total; é necessário decompor a operação em dimensões de tempo, geografia e categoria.

Este notebook estabelece a infraestrutura de dados para a Fase 2. O objetivo é transformar registros brutos de transporte em variáveis estratégicas (como lead time real, custo por rota e desvio de SLA), permitindo que os blocos subsequentes determinem se o modelo operacional atual é um suporte ao crescimento ou um gargalo de valor.

## Dependência
Este notebook **requer** que a Frente Vendas tenha sido executada.
Os arquivos em `data/pre-tratados/` são o ponto de entrada obrigatório.

## Tabelas geradas
| Tabela | Descrição |
|---|---|
| `log_fato.csv` | Fato enriquecido com variáveis logísticas: frete por pedido, rota, período |
| `log_rota.csv` | Agregação por rota (estado_vendedor → estado_cliente): custo, SLA, volume |
| `log_mensal.csv` | Série temporal mensal: frete médio, % frete, lead time, SLA, volume |
| `log_trimestral.csv` | Série trimestral para análises de tendência |
| `log_categoria.csv` | Métricas logísticas por categoria de produto |

---


## PASSO 0 — Verificação de Dependências

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ─── Caminhos — detecta automaticamente a pasta do notebook ──────────────────
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR   = _find_base(NOTEBOOK_DIR)
DIR_PRE    = BASE_DIR / "data" / "pre-tratados"
DIR_LOG    = BASE_DIR / "data" / "logistics"
DIR_LOG.mkdir(parents=True, exist_ok=True)

# ─── Verifica arquivos do Projeto 1 ──────────────────────────────────────────
ARQUIVOS_NECESSARIOS = [
    "fato_vendas.csv", "dim_produto.csv", "dim_cliente.csv",
    "dim_vendedor.csv", "dim_tempo.csv",
]

print("Verificando dependências do Projeto 1...\n")
tudo_ok = True
for arq in ARQUIVOS_NECESSARIOS:
    caminho = DIR_PRE / arq
    if caminho.exists():
        kb = caminho.stat().st_size / 1024
        print(f"  ✅ {arq:<30} {kb:>8.1f} KB")
    else:
        print(f"  ❌ {arq:<30} NÃO ENCONTRADO")
        tudo_ok = False

print()
if tudo_ok:
    print("✅ Todas as dependências encontradas. Pode prosseguir.")
else:
    print("❌ Execute o Projeto 1 (01_ETL_pipeline.ipynb) antes de continuar.")
    print("   Ou copie a pasta data/pre-tratados/ do Projeto 1 para este projeto.")


## PASSO 1 — Carregamento dos Dados do Retail

In [ ]:
def ler_csv(caminho, **kwargs):
    df = pd.read_csv(caminho, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

fato  = ler_csv(DIR_PRE / "fato_vendas.csv")
dim_p = ler_csv(DIR_PRE / "dim_produto.csv")
dim_c = ler_csv(DIR_PRE / "dim_cliente.csv")
dim_v = ler_csv(DIR_PRE / "dim_vendedor.csv")
dim_t = ler_csv(DIR_PRE / "dim_tempo.csv")

# ─── Conversão de tipos ───────────────────────────────────────────────────────
for col in ["data_compra","data_entrega_cliente","data_previsao_entrega",
            "data_aprovacao","data_envio_transportadora"]:
    if col in fato.columns:
        fato[col] = pd.to_datetime(fato[col], errors="coerce")
dim_t["data"] = pd.to_datetime(dim_t["data"], errors="coerce")

for col in ["preco","valor_frete","valor_total_item","valor_pagamento_total",
            "lead_time_dias","atraso_dias","nota_review","numero_parcelas"]:
    if col in fato.columns:
        fato[col] = pd.to_numeric(fato[col], errors="coerce")

# ─── Enriquecimento ───────────────────────────────────────────────────────────
fato = fato.merge(dim_p[["id_produto","nome_categoria_produto"]], on="id_produto", how="left")
fato = fato.merge(dim_c[["id_cliente","estado_cliente"]], on="id_cliente", how="left")
fato = fato.merge(dim_v[["id_vendedor","estado_vendedor"]], on="id_vendedor", how="left")
fato = fato.merge(dim_t[["id_data","ano","mes","trimestre","nome_mes","ano_mes"]],
                  on="id_data", how="left")
fato["periodo"] = fato["ano"].astype(str) + "-Q" + fato["trimestre"].astype(str)

# ─── Filtro temporal (jan/2024 em diante — mesmo padrão do Retail) ───────────
DATA_INICIO = "2024-01-01"
DATA_FIM    = "2025-08-31"   # alinhado com ETL de Vendas
fato = fato[(fato["data_compra"] >= DATA_INICIO) & (fato["data_compra"] <= DATA_FIM)].copy()
fe   = fato[fato["status_pedido"] == "entregue"].copy()

print(f"[FILTRO] Período: {DATA_INICIO} → {DATA_FIM}")
print(f"fato total    : {len(fato):>8} linhas")
print(f"fato entregues: {len(fe):>8} linhas ({len(fe)/len(fato)*100:.1f}%)")


## PASSO 2 — Enriquecimento Logístico

In [ ]:
# ─── Variável de rota: estado_vendedor → estado_cliente ──────────────────────
fe["rota"] = fe["estado_vendedor"].fillna("??") + " → " + fe["estado_cliente"].fillna("??")

# ─── Frete como % do preço do produto (barreira de conversão) ────────────────
fe["pct_frete_preco"]    = fe["valor_frete"] / fe["preco"].replace(0, np.nan) * 100
fe["frete_por_pedido"]   = fe["valor_frete"]   # alias semântico para clareza
fe["ticket_com_frete"]   = fe["preco"] + fe["valor_frete"]

# ─── Classificação de entrega ────────────────────────────────────────────────
fe["entregue_no_prazo"] = pd.to_numeric(fe["entregue_no_prazo"], errors="coerce")
fe["atrasado"]          = (fe["atraso_dias"] > 0).astype(int)
fe["atraso_grave"]      = (fe["atraso_dias"] > 7).astype(int)

# ─── Custo implícito de SLA (herda correlação do Retail) ─────────────────────
# Nota: o valor de referência será calculado dinamicamente nos notebooks de análise
# a partir da regressão lead_time × nota_review

print("Variáveis logísticas criadas:")
print(f"  pct_frete_preco   : frete como % do preço pago pelo cliente")
print(f"  frete_por_pedido  : valor absoluto do frete cobrado")
print(f"  ticket_com_frete  : preço + frete (custo total percebido pelo cliente)")
print(f"  rota              : {fe['rota'].nunique()} rotas únicas (estado_vendedor → estado_cliente)")
print(f"  atrasado          : {fe['atrasado'].sum():,} pedidos com atraso")
print(f"  atraso_grave      : {fe['atraso_grave'].sum():,} pedidos com atraso > 7 dias")


## PASSO 3 — Geração das Tabelas Analíticas

In [ ]:
# ─── log_mensal: série temporal para tendência e custo de inação ─────────────
log_mensal = (
    fe.groupby("ano_mes")
    .agg(
        n_pedidos        = ("id_pedido",         "nunique"),
        frete_total      = ("valor_frete",        "sum"),
        frete_medio      = ("valor_frete",        "mean"),
        receita_total    = ("preco",              "sum"),
        ticket_medio     = ("preco",              "mean"),
        lead_medio       = ("lead_time_dias",     "mean"),
        lead_p90         = ("lead_time_dias",     lambda x: x.quantile(0.9)),
        pct_no_prazo     = ("entregue_no_prazo",  "mean"),
        nota_media       = ("nota_review",        "mean"),
        n_atrasados      = ("atrasado",           "sum"),
    )
    .reset_index().sort_values("ano_mes")
)
log_mensal["pct_frete_receita"] = log_mensal["frete_total"] / log_mensal["receita_total"] * 100
log_mensal["pct_no_prazo"]      = pd.to_numeric(log_mensal["pct_no_prazo"], errors="coerce") * 100
log_mensal["pct_atrasado"]      = log_mensal["n_atrasados"] / log_mensal["n_pedidos"] * 100
log_mensal["crescimento_mom"]   = log_mensal["frete_medio"].pct_change() * 100

# ─── log_trimestral: série trimestral ────────────────────────────────────────
log_trim = (
    fe.groupby("periodo")
    .agg(
        n_pedidos     = ("id_pedido",        "nunique"),
        frete_total   = ("valor_frete",       "sum"),
        frete_medio   = ("valor_frete",       "mean"),
        receita_total = ("preco",             "sum"),
        lead_medio    = ("lead_time_dias",    "mean"),
        pct_no_prazo  = ("entregue_no_prazo", "mean"),
        nota_media    = ("nota_review",       "mean"),
    )
    .reset_index().sort_values("periodo")
)
log_trim["pct_frete_receita"] = log_trim["frete_total"] / log_trim["receita_total"] * 100
log_trim["pct_no_prazo"]      = pd.to_numeric(log_trim["pct_no_prazo"], errors="coerce") * 100

# ─── log_rota: métricas por rota geográfica ──────────────────────────────────
log_rota = (
    fe.groupby(["estado_vendedor","estado_cliente","rota"])
    .agg(
        n_pedidos     = ("id_pedido",        "nunique"),
        frete_total   = ("valor_frete",       "sum"),
        frete_medio   = ("valor_frete",       "mean"),
        receita_total = ("preco",             "sum"),
        lead_medio    = ("lead_time_dias",    "mean"),
        pct_no_prazo  = ("entregue_no_prazo", "mean"),
        nota_media    = ("nota_review",       "mean"),
        n_atrasados   = ("atrasado",          "sum"),
    )
    .reset_index().sort_values("receita_total", ascending=False)
)
log_rota["pct_frete_receita"] = log_rota["frete_total"] / log_rota["receita_total"] * 100
log_rota["pct_no_prazo"]      = pd.to_numeric(log_rota["pct_no_prazo"], errors="coerce") * 100
log_rota["pct_receita"]       = log_rota["receita_total"] / log_rota["receita_total"].sum() * 100
log_rota["pct_acum"]          = log_rota["pct_receita"].cumsum()

# ─── log_categoria: métricas por categoria ───────────────────────────────────
log_cat = (
    fe.groupby("nome_categoria_produto")
    .agg(
        n_pedidos     = ("id_pedido",        "nunique"),
        frete_total   = ("valor_frete",       "sum"),
        frete_medio   = ("valor_frete",       "mean"),
        receita_total = ("preco",             "sum"),
        lead_medio    = ("lead_time_dias",    "mean"),
        pct_no_prazo  = ("entregue_no_prazo", "mean"),
        nota_media    = ("nota_review",       "mean"),
    )
    .reset_index().sort_values("receita_total", ascending=False)
)
log_cat["pct_frete_receita"] = log_cat["frete_total"] / log_cat["receita_total"] * 100
log_cat["pct_no_prazo"]      = pd.to_numeric(log_cat["pct_no_prazo"], errors="coerce") * 100
log_cat["pct_receita"]       = log_cat["receita_total"] / log_cat["receita_total"].sum() * 100

# ─── log_fato: fato enriquecido ──────────────────────────────────────────────
log_fato = fe[[
    "id_pedido","id_produto","id_cliente","id_vendedor","id_data",
    "data_compra","data_entrega_cliente","data_previsao_entrega",
    "preco","valor_frete","frete_por_pedido","pct_frete_preco",
    "ticket_com_frete","lead_time_dias","atraso_dias","atrasado",
    "atraso_grave","entregue_no_prazo","nota_review","status_pedido",
    "nome_categoria_produto","estado_cliente","estado_vendedor",
    "rota","ano","mes","trimestre","nome_mes","ano_mes","periodo"
]].copy()

print("Tabelas geradas:")
print(f"  log_fato        : {len(log_fato):>8} linhas | {len(log_fato.columns)} colunas")
print(f"  log_mensal      : {len(log_mensal):>8} linhas")
print(f"  log_trim        : {len(log_trim):>8} linhas")
print(f"  log_rota        : {len(log_rota):>8} linhas | {log_rota['rota'].nunique()} rotas")
print(f"  log_cat         : {len(log_cat):>8} linhas | {log_cat['nome_categoria_produto'].nunique()} categorias")


## PASSO 4 — Exportação

In [ ]:
tabelas = {
    "log_fato.csv"        : log_fato,
    "log_mensal.csv"      : log_mensal,
    "log_trimestral.csv"  : log_trim,
    "log_rota.csv"        : log_rota,
    "log_categoria.csv"   : log_cat,
}

for nome, df in tabelas.items():
    df.to_csv(DIR_LOG / nome, index=False, encoding="utf-8", sep=",", decimal=".")
    print(f"  ✅ {nome:<25} → {len(df):>6} linhas")

print(f"\nExportado em: {DIR_LOG}")
print("\nETL Logistics concluído. Execute os notebooks de análise em sequência.")


---
*Próximo notebook: `01_diagnostico_modelo_atual.ipynb` — O modelo terceirizado já chegou no seu limite?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
